# PADAssembler visual check — all four detectors, unified pathway

`notebooks/detector_assembly_crosscheck.ipynb` (2026-09-16) compared our assembly
against Resonet's independent CrystFEL-based assembly, but at the time Jungfrau 4M
was **not** assembled by Reborn at all — it was a bare `_to_2d(raw)` passthrough of
the pre-assembled LCLS canvas, while AGIPD/ePix10k/Eiger4M already went through
`PADAssembler`. That crosscheck notebook's `assemble_ours()` still special-cases
Jungfrau with `_to_2d`, so it never exercised the real pathway for that detector.

Since then (plan `2026-09-24-jungfrau-pad-assembler`, merged via PR #37), Jungfrau 4M
was switched onto the same `get_geometry()` → `get_assembler()` →
`src.preprocessing.pipeline.assemble_only()` pathway as the other three detectors,
using `jungfrau_8_pad_geometry_list()` (8 panels, 103mm) with real panel placement
instead of the raw canvas's arbitrary layout. That switch was verified structurally
(shape, mask coverage, bit-exactness of the *pipeline*) but — per kanban item
"Visually verify fixed panel layouts (reopened — prior verdict unconfirmed)" —
never re-confirmed visually.

**This notebook's job:** assemble all four detectors through the exact current
production pathway (`assemble_only()` + `valid_pixel_mask()`, the same functions the
training pipeline calls) and render them so panel placement, orientation, and gap
pattern can be eyeballed directly. Jungfrau is the detector of interest; the other
three are included as a sanity baseline (their pathway hasn't changed since the prior
crosscheck).

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# hdf5plugin MUST be imported before h5py to register the bitshuffle filter.
import hdf5plugin  # noqa: F401
import h5py

from src.preprocessing.geometry import get_assembler, get_geometry
from src.preprocessing.pipeline import assemble_only, valid_pixel_mask
from src.preprocessing.io import read_detector_description, read_frame

DATA_ROOT = "/data/bioxfel/user/gihan/Resonet/production"
LABEL_KEY = "entry_1/data_1/data"

PRODUCTION_FILES = {
    "AGIPD": os.path.join(DATA_ROOT, "agipd_20k", "compressed0.cxi"),
    "JUNGFRAU_4M": os.path.join(DATA_ROOT, "jungfrau_20k", "compressed0.cxi"),
    "ePix10k": os.path.join(DATA_ROOT, "epix10k_20k", "compressed0.cxi"),
    "Eiger4M": os.path.join(DATA_ROOT, "eiger4m_20k", "compressed0.cxi"),
}
for label, path in PRODUCTION_FILES.items():
    print(f"{label:12s} exists={os.path.exists(path)}  {path}")

## 1. Production assembly path

This is the *actual* training pathway, not a reimplementation: `assemble_only()` and
`valid_pixel_mask()` are imported directly from `src.preprocessing.pipeline` (the same
module `AsymmetricCXIDataset` and the eval benchmark call). No copy, no second
implementation to drift out of sync.

In [ ]:
_pair_cache = {}  # (detector, frame_idx) -> (assembled, mask, desc)
_nframes_cache = {}


def n_frames(name):
    if name not in _nframes_cache:
        with h5py.File(PRODUCTION_FILES[name], "r") as f:
            _nframes_cache[name] = int(f[LABEL_KEY].shape[0])
    return _nframes_cache[name]


def load_assembled(name, idx):
    key = (name, idx)
    if key in _pair_cache:
        return _pair_cache[key]

    cxi = PRODUCTION_FILES[name]
    raw = np.asarray(read_frame(cxi, idx), dtype=np.float32)
    desc = read_detector_description(cxi)

    pads = get_geometry(desc)
    assembler = get_assembler(desc)
    assembled = assemble_only(raw, pads, desc, assembler=assembler)
    mask = valid_pixel_mask(desc)

    meta = {
        "desc": desc,
        "raw_shape": raw.shape,
        "n_panels": len(pads),
        "assembled_shape": assembled.shape,
        "valid_frac": float(mask.mean()),
    }
    result = (assembled, mask, meta)
    _pair_cache[key] = result
    return result

## 2. Interactive viewer

Gaps/padding (`mask == False`) are tinted cyan on top of the intensity image so panel
boundaries are visible even on a sparse/low-signal frame. A scrambled, rotated, or
mirrored panel layout shows up as a gap pattern that doesn't form a sensible detector
shape (e.g. panels not aligned into rows/quadrants).

In [ ]:
w_detector = widgets.Dropdown(
    options=list(PRODUCTION_FILES.keys()), value="JUNGFRAU_4M", description="detector:"
)
w_frame = widgets.IntSlider(value=0, min=0, max=0, step=1, description="frame:")
w_out = widgets.Output()


def _refresh_frame_bounds(*_):
    w_frame.max = max(0, n_frames(w_detector.value) - 1)


def _render(*_):
    with w_out:
        w_out.clear_output(wait=True)
        assembled, mask, meta = load_assembled(w_detector.value, w_frame.value)

        pos = assembled[assembled > 0]
        vmin = float(np.percentile(pos, 1)) if len(pos) else 0.0
        vmax = float(np.percentile(pos, 99.5)) if len(pos) else 1.0

        fig, ax = plt.subplots(figsize=(7, 7), dpi=110)
        ax.imshow(assembled, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
        # Cyan overlay on invalid (gap/padding) pixels, alpha-blended.
        overlay = np.zeros((*mask.shape, 4))
        overlay[~mask] = (0.0, 1.0, 1.0, 0.35)
        ax.imshow(overlay, origin="upper")
        ax.set_title(
            f"{meta['desc']} — frame {w_frame.value}\n"
            f"raw={meta['raw_shape']}  panels={meta['n_panels']}  "
            f"assembled={meta['assembled_shape']}  valid_frac={meta['valid_frac']:.3f}",
            fontsize=9,
        )
        ax.axis("off")
        plt.show()


w_detector.observe(lambda c: (_refresh_frame_bounds(), _render()), names="value")
w_frame.observe(_render, names="value")
_refresh_frame_bounds()
display(widgets.VBox([widgets.HBox([w_detector, w_frame]), w_out]))
_render()

## 3. Static summary gallery (frame 0, all four detectors)

Fixed, no widgets, so it survives a headless `nbconvert --execute` run and can be
diffed/archived.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 11), dpi=130)
for ax, name in zip(axes.ravel(), PRODUCTION_FILES.keys()):
    assembled, mask, meta = load_assembled(name, 0)
    pos = assembled[assembled > 0]
    vmin = float(np.percentile(pos, 1)) if len(pos) else 0.0
    vmax = float(np.percentile(pos, 99.5)) if len(pos) else 1.0

    ax.imshow(assembled, cmap="inferno", vmin=vmin, vmax=vmax, origin="upper")
    overlay = np.zeros((*mask.shape, 4))
    overlay[~mask] = (0.0, 1.0, 1.0, 0.35)
    ax.imshow(overlay, origin="upper")
    ax.set_title(
        f"{meta['desc']}\n{meta['assembled_shape']}, {meta['n_panels']} panels, "
        f"valid={meta['valid_frac']:.2f}",
        fontsize=10,
    )
    ax.axis("off")
fig.suptitle("Unified PADAssembler pathway — frame 0, all four detectors", fontsize=13)
fig.tight_layout()

out_dir = os.path.join(PROJECT_ROOT, "docs", "figures", "assembly_crosscheck")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "pad_assembler_unified_frame0.png")
fig.savefig(out_path, bbox_inches="tight")
print(f"saved → {out_path}")
plt.show()